# Gold Yahoo: multi-horizon Transformer-MAF training

This isolated notebook imports `fixed_notebooks/project.zip` without modifying it. Each daily target is a three-dimensional log-return vector: `[1-day, 3-day, 7-day]`. The multivariate Transformer-MAF learns their joint conditional distribution and exports artifacts consumed by the Streamlit fan-chart app.

In [ ]:
%pip -q install yfinance pyyaml orjson

from google.colab import drive, files
from pathlib import Path
import json, random, shutil, sys, zipfile

drive.mount('/content/drive')
PROJECT_ZIP = Path('/content/drive/MyDrive/fixed_notebooks/project.zip')
if not PROJECT_ZIP.exists():
    print('Upload fixed_notebooks/project.zip')
    uploaded = files.upload()
    PROJECT_ZIP = Path(next(iter(uploaded)))

WORK_ROOT = Path('/content/gold_yahoo_work')
PROJECT_ROOT = WORK_ROOT / 'project'
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
with zipfile.ZipFile(PROJECT_ZIP) as archive:
    archive.extractall(WORK_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Project imported from:', PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import torch
import yfinance as yf
from torch.utils.data import DataLoader

from flowforecast.data.covariates import time_features
from flowforecast.data.windowing import RollingWindowDataset, WindowedDataset
from flowforecast.training.trainer import Trainer
from flowforecast.utils.build import build_model, count_parameters

SEED = 42
TICKER = 'GC=F'
RETURN_HORIZONS = [1, 3, 7]
PREDICTION_LENGTH = 10
CONTEXT_LENGTH = 30
LAGS = [1, 2]
EPOCHS = 20
BATCH_SIZE = 64
BATCHES_PER_EPOCH = 100
VALIDATION_WINDOWS = 5
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

raw = yf.download(TICKER, period='max', interval='1d', auto_adjust=False, progress=False, threads=False)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
close = raw['Close'].astype(float).rename('Close').dropna()
close.index = pd.to_datetime(close.index).tz_localize(None)
close = close[~close.index.duplicated(keep='last')]
return_frame = pd.DataFrame({f'return_{h}d': np.log(close / close.shift(h)) for h in RETURN_HORIZONS}).dropna().astype('float32')
print(f'prices: {len(close):,}; aligned return rows: {len(return_frame):,}; range: {return_frame.index.min().date()} -> {return_frame.index.max().date()}')
display(return_frame.tail())

In [ ]:
# Reuse the unchanged project windowing, scaling, Transformer, MAF, and Trainer code.
series = torch.from_numpy(return_frame.to_numpy()).t().contiguous()
tf = time_features(pd.DatetimeIndex(return_frame.index), 'B')
validation_end = len(return_frame)
training_end = validation_end - VALIDATION_WINDOWS * PREDICTION_LENGTH
if training_end <= CONTEXT_LENGTH + max(LAGS):
    raise ValueError('Not enough Yahoo Finance history for the requested windows')
train_dataset = WindowedDataset(series=series, time_feat=tf, context_length=CONTEXT_LENGTH, prediction_length=PREDICTION_LENGTH, lags=LAGS, num_windows_per_epoch=BATCH_SIZE * BATCHES_PER_EPOCH, end_index=training_end)
val_dataset = RollingWindowDataset(series=series, time_feat=tf, context_length=CONTEXT_LENGTH, prediction_length=PREDICTION_LENGTH, lags=LAGS, num_windows=VALIDATION_WINDOWS, end_index=validation_end)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

CONFIG = {
    'dataset': {'name': 'gold_yahoo_multi_horizon', 'freq': 'B', 'num_series': 3, 'prediction_length': PREDICTION_LENGTH, 'context_length': CONTEXT_LENGTH, 'lags': LAGS, 'dequantize': False},
    'flow': {'type': 'maf', 'num_blocks': 5, 'hidden': 100, 'batch_norm': True, 'bn_min_var': 1e-4, 'maf_log_scale_bound': 3.0, 'maf_mu_bound': 10.0},
    'backbone': {'type': 'transformer', 'd_model': 32, 'n_heads': 8, 'num_encoder_layers': 3, 'num_decoder_layers': 3, 'dropout': 0.1},
    'model': {'conditioning_length': 200, 'condition_bound': 10.0},
    'training': {'lr': 1e-3, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'batches_per_epoch': BATCHES_PER_EPOCH, 'maf_grad_clip': 5.0, 'seed': SEED, 'val_windows': VALIDATION_WINDOWS, 'select_metric': 'last', 'validation_num_samples': 100, 'validation_seed': 1729, 'validation_interval': 5, 'lr_schedule': 'onecycle', 'max_lr': 3e-3},
    'evaluation': {'num_samples': 500, 'num_test_windows': VALIDATION_WINDOWS},
}
cond_size = int(tf.shape[0] + series.shape[0] * len(LAGS))
model = build_model(CONFIG, num_series=series.shape[0], cond_size=cond_size, context_length=CONTEXT_LENGTH).to(DEVICE)
print(type(model).__name__, count_parameters(model), 'parameters')

In [ ]:
CHECKPOINT_DIR = Path('/content/gold_yahoo_artifacts/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
trainer = Trainer(model, train_loader, val_loader=val_loader, device=DEVICE, lr=1e-3, grad_clip=5.0, batches_per_epoch=BATCHES_PER_EPOCH, select_metric='last', validation_num_samples=100, validation_seed=1729, validation_interval=5, lr_schedule='onecycle', max_lr=3e-3)
history = trainer.fit(num_epochs=EPOCHS, checkpoint_dir=str(CHECKPOINT_DIR), run_name='gold_yahoo_transformer_maf', extra_state={'config': CONFIG, 'seed': SEED, 'baseline': None, 'num_series': 3, 'cond_size': cond_size, 'context_length': CONTEXT_LENGTH, 'prediction_length': PREDICTION_LENGTH, 'lags': LAGS, 'freq': 'B', 'model_contract': 'tempflow_v5_official_parity'})

export_root = Path('/content/gold_yahoo_artifacts')
export_path = export_root / 'gold_yahoo_model.pt'
metadata_path = export_root / 'gold_yahoo_metadata.json'
final_checkpoint = CHECKPOINT_DIR / 'gold_yahoo_transformer_maf.pt'
shutil.copy2(final_checkpoint, export_path)
metadata = {'model_name': 'Transformer-MAF', 'ticker': TICKER, 'target': 'multi_horizon_log_return', 'return_horizons': RETURN_HORIZONS, 'series_labels': [f'{h}-day return' for h in RETURN_HORIZONS], 'freq': 'B', 'num_series': 3, 'cond_size': cond_size, 'context_length': CONTEXT_LENGTH, 'prediction_length': PREDICTION_LENGTH, 'lags': LAGS, 'checkpoint_selection': 'last_epoch', 'training_epochs': EPOCHS, 'training_end_date': str(return_frame.index[training_end - 1].date()), 'validation_end_date': str(return_frame.index[validation_end - 1].date()), 'config': CONFIG, 'model_contract': 'tempflow_v5_official_parity'}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
drive_output = Path('/content/drive/MyDrive/gold_yahoo_artifacts')
drive_output.mkdir(parents=True, exist_ok=True)
shutil.copy2(export_path, drive_output / export_path.name)
shutil.copy2(metadata_path, drive_output / metadata_path.name)
print('Exported:', export_path, metadata_path)
print('Drive artifacts:', drive_output)